In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision.transforms import ToTensor
from torchvision import models
from torch.utils.data import WeightedRandomSampler

np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# data augmentation pipeline
import torchvision.transforms as transforms
from torchvision import datasets

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(30),
    transforms.RandomVerticalFlip(),
    transforms.Resize((224, 224))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

train_set = datasets.ImageFolder(root = 'Splitted2D/train', transform = train_transform)
val_set = datasets.ImageFolder(root = 'Splitted2D/val', transform = val_transform)
test_set = datasets.ImageFolder(root = 'Splitted2D/test', transform = test_transform)

In [ ]:
idx2class = {v: k for k, v in train_set.class_to_idx.items()}

def get_class_distribution(dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
print("Distribution of classes: \n", get_class_distribution(train_set))

In [ ]:
BATCH_SIZE = 32

def get_class_distribution_loaders(dataloader_obj, dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for _,j in dataloader_obj:
        y_idx = j.item()
        y_lbl = idx2class[y_idx]
        count_dict[str(y_lbl)] += 1
            
    return count_dict

target_list = torch.tensor(train_set.targets)
class_count = [i for i in get_class_distribution(train_set).values()]
class_weights = 1./torch.tensor(class_count, dtype=torch.float) 
print(class_weights)
###################### OUTPUT ######################
#tensor([0.0068, 0.0054, 0.0114, 0.0123])

class_weights_all = class_weights[target_list]

weighted_sampler = WeightedRandomSampler(
    weights=class_weights_all,
    num_samples=len(class_weights_all),
    replacement=False
)

train_loader = DataLoader(dataset = train_set, shuffle = False, batch_size = BATCH_SIZE, sampler = weighted_sampler)
val_loader = DataLoader(dataset = val_set, shuffle = True, batch_size = BATCH_SIZE)
test_loader = DataLoader(dataset = test_set, shuffle = True, batch_size = BATCH_SIZE)

In [ ]:
resnet18 = models.resnet50(weights = 'IMAGENET1K_V1')
count = 0
# for child in resnet18.children():
#     print(count, child)
#     count += 1
#     if (count < 9):
#         for param in child.parameters():
#             param.requires_grad = False

resnet18.fc = nn.Sequential(
    #nn.BatchNorm2d(512),  
    nn.Dropout(0.2, inplace = False),
    nn.Linear(2048, 1024, bias = True),
    nn.LeakyReLU(),
    nn.Dropout(0.2, inplace = False),
    nn.Linear(1024, 512, bias = True),
    nn.LeakyReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256, bias = True),
    nn.LeakyReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 3, bias = True)
)

In [ ]:
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
from torchsummary import summary
#summary(resnet18, (3, 224, 224))
N_EPOCHS = 25
LR = 0.01
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet18.parameters(), lr= LR) 
resnet18 = resnet18.to(device)


num_epochs = 50
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, val_loader, resnet18, loss, optimizer)


# for t in range(25):
#     print(f"Epoch {t+1}\n-------------------------------")
#     train_loop(train_loader, resnet18, loss, optimizer)


In [ ]:
for epoch in trange(N_EPOCHS, desc = "TRAINING"):
    correct, total = 0, 0
    train_loss = 0.0
    
    train_correct, train_total = 0, 0
    for batch in tqdm(train_loader, desc = f"EPOCH {epoch + 1} IN TRAINING", leave = False):
        x, y = batch
        #print(x.shape)
        x, y = x.to(device), y.to(device)
        y_hat = resnet18(x)
        
        loss = criterion(y_hat, y)
        
        train_loss += loss.detach().cpu().item() / len(train_loader)
        train_correct += torch.sum(torch.argmax(y_hat, dim = 1) == y).detach().cpu().item()
        train_total += len(x)
        
        optimizer.zero_grad()
        #count = 0
#         for name, param in resnet18.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                 print(gradient_norm)
                #count += 1
        
        loss.backward()
        optimizer.step()
    print(f"Accuracy: {train_correct / train_total * 100:.2f}%")
        #print(torch.argmax(y_hat, dim = 1))
    for batch in tqdm(val_loader, desc = 'VALIDATION'):
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        y_hat = resnet18(x)
        
        correct += torch.sum(torch.argmax(y_hat, dim = 1) == y).detach().cpu().item()
        total += len(x)
    
    print(f"EPOCH {epoch + 1} / {N_EPOCHS} loss: {train_loss:.2f} accuracy: {train_correct / train_total * 100:.2f}%")
    print(f"Validation accuracy: {correct / total * 100:.2f}%")

with torch.no_grad():
    correct, total = 0, 0
    test_loss = 0.0
    for batch in tqdm(test_loader, desc = "TESTING"):
        x, y = batch

        x, y = x.to(device), y.to(device)
        
        y_hat = resnet18(x.float())

        loss = criterion(y_hat, y)

        test_loss += loss.detach().cpu().item() / len(test_loader)

        correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
        total += len(x)
    print(f"Test loss: {test_loss:.2f}")
    print(f"Test accuracy: {correct / total * 100:.2f}%")
    